# === ЭТАП 3 ===

In [1]:
import logging
import sys

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

log_file = logging.FileHandler('recommendations.log', mode='w')
log_file.setFormatter(logging.Formatter('%(asctime)s %(levelname)s %(message)s'))

console_out = logging.StreamHandler(sys.stdout)
console_out.setFormatter(logging.Formatter('%(message)s'))

logging.basicConfig(handlers=(log_file, console_out), level=logging.INFO)

# Загрузка данных

Если необходимо, то загружаем items.parquet, events.parquet.

In [10]:
items = pd.read_parquet('../data/items.parquet')
events = pd.read_parquet('../data/events.parquet')

events = events.merge(items.query('type == "track"')[['item_id', 'votes', 'rating']], on='item_id')

In [11]:
import sklearn.preprocessing

# перекодируем идентификаторы пользователей: 
# из имеющихся в последовательность 0, 1, 2, ...
user_encoder = sklearn.preprocessing.LabelEncoder()
user_encoder.fit(events['user_id'])
events['user_id_enc'] = user_encoder.transform(events['user_id'])

# перекодируем идентификаторы объектов: 
# из имеющихся в последовательность 0, 1, 2, ...
item_encoder = sklearn.preprocessing.LabelEncoder()
item_encoder.fit(items['item_id'])
items['item_id_enc'] = item_encoder.transform(items['item_id'])
events['item_id_enc'] = item_encoder.transform(events['item_id'])

# Разбиение данных

Разбиваем данные на тренировочную, тестовую выборки.

In [12]:
train_split_date = pd.to_datetime('2022-12-16').to_datetime64()

train_split_date_idx = events['started_at'] < train_split_date

events_train = events[train_split_date_idx]
events_test = events[~train_split_date_idx]

events_train.shape, events_test.shape

((9110491, 9), (3912838, 9))

In [13]:
# количество пользователей в train и test
users_train = events_train['user_id'].drop_duplicates()
users_test = events_test['user_id'].drop_duplicates()
# количество пользователей, которые есть и в train, и в test
common_users = set(users_train) & set(users_test)

len(users_train), len(users_test), len(common_users)

(1113710, 702781, 456576)

# Похожие

Рассчитаем похожие, они позже пригодятся для онлайн-рекомендаций.

In [14]:
import scipy
from implicit.als import AlternatingLeastSquares

def get_als_model(data, row, col):

    # создаём sparse-матрицу формата CSR 
    matrix = scipy.sparse.csr_matrix((
        data,
        (row, col)),
        dtype=np.int8)
    
    model = AlternatingLeastSquares(factors=50, iterations=10, regularization=0.05, random_state=0)
    model.fit(matrix) 
    
    return model

In [15]:
def get_similar_items(model, item_ids_enc, max_similar_items = 10):

    # получаем списки похожих объектов, используя ранее полученную ALS-модель
    # метод similar_items возвращает и сам объект, как наиболее похожий
    # этот объект мы позже отфильтруем, но сейчас запросим на 1 больше
    similar_items = model.similar_items(item_ids_enc, N=max_similar_items+1)

    # преобразуем полученные списки в табличный формат
    sim_item_item_ids_enc = similar_items[0]
    sim_item_scores = similar_items[1]

    similar_items = pd.DataFrame({
        "item_id_enc": item_ids_enc,
        "sim_item_id_enc": sim_item_item_ids_enc.tolist(), 
        "score": sim_item_scores.tolist()})
    similar_items = similar_items.explode(["sim_item_id_enc", "score"], ignore_index=True)

    # приводим типы данных
    similar_items["sim_item_id_enc"] = similar_items["sim_item_id_enc"].astype("int")
    similar_items["score"] = similar_items["score"].astype("float")

    # получаем изначальные идентификаторы
    similar_items["item_id_1"] = item_encoder.inverse_transform(similar_items["item_id_enc"])
    similar_items["item_id_2"] = item_encoder.inverse_transform(similar_items["sim_item_id_enc"])
    similar_items = similar_items.drop(columns=["item_id_enc", "sim_item_id_enc"])

    # убираем пары с одинаковыми объектами
    similar_items = similar_items.query("item_id_1 != item_id_2")
    
    return similar_items

In [16]:
def print_sim_items(item_id, similar_items):

    item_columns_to_use = ['item_id', 'name', 'votes', 'rating']
    
    item_id_1 = items.query('item_id == @item_id')[item_columns_to_use]
    display(item_id_1)
    
    si = similar_items.query('item_id_1 == @item_id')
    si = si.merge(items[item_columns_to_use].set_index('item_id'), left_on='item_id_2', right_index=True)
    display(si)

In [17]:
df_events = events_train[['rating', 'item_id', 'user_id']].rename(columns={'user_id': 'id'})
df_events['id'] = 'track_' + df_events['id'].astype(str)

df_items  = items[items['item_id'].isin(df_events['item_id'].unique())]

In [18]:
df_album = pd.DataFrame(df_items['album'].apply(eval).explode('album').tolist()).rename(columns={'track_id': 'item_id'})[['item_id', 'rating', 'id']]
df_album['id'] = 'album_' + df_album['id'].astype(str)

df_artist = pd.DataFrame(df_items['artist'].apply(eval).explode('artist').tolist()).rename(columns={'track_id': 'item_id'})[['item_id', 'rating', 'id']]
df_artist['id'] = 'artist_' + df_artist['id'].astype(str)

df_genre = pd.DataFrame(df_items['genre'].apply(eval).explode('genre').tolist()).rename(columns={'track_id': 'item_id'})[['item_id', 'rating', 'id']]
df_genre['id'] = 'genre_' + df_genre['id'].astype(str)


KeyError: 'album'

In [11]:
df = pd.concat([
    df_events,
    df_album,
    df_artist,
    df_genre
])

df['item_id_enc'] = item_encoder.transform(df['item_id'])

encoder = sklearn.preprocessing.LabelEncoder()
encoder.fit(df['id'])

df['id_enc'] = encoder.transform(df['id'])


In [19]:
df.query('item_id == 53404')

,rating,item_id,id,item_id_enc,id_enc
5615721,10.00,53404,track_592791,701,805626
5288603,10.00,53404,track_558272,701,774471
8829292,10.00,53404,track_931910,701,1111183
3422103,10.00,53404,track_361278,701,597309
5189866,10.00,53404,track_547834,701,765145
...,...,...,...,...,...
4,7.67,53404,album_294912,701,20193
0,8.57,53404,artist_9262,701,58231
0,9.32,53404,genre_102,701,58713
1,8.20,53404,genre_14,701,58729


In [13]:
als_model = get_als_model(df['rating'], df['id_enc'], df['item_id_enc'])
similar_items_users = get_similar_items(als_model, df['item_id_enc'].unique())

/home/mle-user/mle_projects/mle-project-sprint-4-v001/env_recsys_start/lib/python3.10/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 4 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 10/10 [00:46<00:00,  4.70s/it]


In [21]:
print_sim_items(35505245, similar_items_users)

,item_id,name,votes,rating
19154,35505245,I Got Love,99490,9.78


,score,item_id_1,item_id_2,name,votes,rating
34,0.914316,35505245,35780252,Заплаканная,15615,6.78
35,0.907032,35505245,35476145,"Дико, например",15411,6.76
36,0.847588,35505245,35869002,Именно та,8747,6.04
37,0.845927,35505245,35554093,Yellow Light,1168,4.05
38,0.838624,35505245,34187641,Ламбада,6994,5.78
39,0.837808,35505245,34972337,Монетка,19185,7.06
40,0.832397,35505245,36519033,DLBM,14730,6.70
41,0.828502,35505245,35419758,Цепи,3563,5.06
42,0.826907,35505245,34362335,Танцуйте,24060,7.38
43,0.820018,35505245,35674500,Back2Leto,1935,4.48


In [14]:
raise

RuntimeError: No active exception to reraise

In [10]:
als_model = get_als_model(df_events['rating'], df_events['user_id_enc'], df_events['item_id_enc'])
similar_items_users = get_similar_items(als_model, df_events['item_id_enc'].unique())

/home/mle-user/mle_projects/mle-project-sprint-4-v001/env_recsys_start/lib/python3.10/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 4 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 10/10 [00:44<00:00,  4.47s/it]


In [11]:
print_sim_items(787433, similar_items_users)

,item_id,name,votes,rating
4607,787433,Far From Over,1000,3.93


,score,item_id_1,item_id_2,name,votes,rating
2031994,0.937474,787433,1536466,Na Na Na (Na Na Na Na Na Na Na Na Na),7845,5.91
2031995,0.933368,787433,5706316,Stray Heart,1649,4.34
2031996,0.932947,787433,3480521,Everybody Talks,11366,6.36
2031997,0.931502,787433,6343854,Things We Lost in the Fire,3611,5.07
2031998,0.925710,787433,2216681,Get Up!,2036,4.53
2031999,0.923320,787433,2688595,Dreaming,1424,4.22
2032000,0.922266,787433,789166,Slam,1125,4.02
2032001,0.917639,787433,675181,Supernatural,1009,3.94
2032002,0.917027,787433,2178469,Sick,2176,4.59
2032003,0.916919,787433,4786230,Kill the DJ,3055,4.91


In [ ]:
raise

In [12]:
df_genre = pd.DataFrame(df_items['genre'].apply(eval).explode('genre').tolist()).rename(columns={'track_id': 'item_id'})

genre_encoder = sklearn.preprocessing.LabelEncoder()
genre_encoder.fit(df_genre['id'])

df_genre['id_enc'] = genre_encoder.transform(df_genre['id'])
df_genre['item_id_enc'] = item_encoder.transform(df_genre['item_id'])

als_model = get_als_model(df_genre['rating'], df_genre['id_enc'], df_genre['item_id_enc'])
similar_items_genre = get_similar_items(als_model, df_genre['item_id_enc'].unique())

100%|██████████| 10/10 [00:01<00:00,  9.17it/s]


In [13]:
print_sim_items(787433, similar_items_genre)

,item_id,name,votes,rating
4607,787433,Far From Over,1000,3.93


,score,item_id_1,item_id_2,name,votes,rating
2031994,1.0,787433,58623710,Drowning,1367,4.18
2031995,1.0,787433,67110388,Рассвет,2835,4.83
2031996,1.0,787433,46156705,Me and the Devil,1142,4.04
2031997,1.0,787433,41535531,Despicable,6461,5.69
2031998,1.0,787433,39803206,Wicked Gonna Come,1541,4.28
2031999,1.0,787433,28431553,Line It Up,2309,4.64
2032000,1.0,787433,58682892,Ключи,1715,4.38
2032001,1.0,787433,52846288,Хорошие девочки,1455,4.24
2032002,1.0,787433,39412264,Гореть,3338,4.99
2032003,1.0,787433,38476760,Сядь и успокойся,1770,4.40


In [14]:
df_artist = pd.DataFrame(df_items['artist'].apply(eval).explode('artist').tolist()).rename(columns={'track_id': 'item_id'})

artist_encoder = sklearn.preprocessing.LabelEncoder()
artist_encoder.fit(df_artist['id'])

df_artist['id_enc'] = artist_encoder.transform(df_artist['id'])
df_artist['item_id_enc'] = item_encoder.transform(df_artist['item_id'])

als_model = get_als_model(df_artist['rating'], df_artist['id_enc'], df_artist['item_id_enc'])
similar_items_artist = get_similar_items(als_model, df_artist['item_id_enc'].unique())

100%|██████████| 10/10 [00:01<00:00,  7.48it/s]


In [15]:
print_sim_items(787433, similar_items_artist)

,item_id,name,votes,rating
4607,787433,Far From Over,1000,3.93


,score,item_id_1,item_id_2,name,votes,rating
2031994,0.878050,787433,59580150,Не отпускай,3893,5.15
2031995,0.873845,787433,29133043,Пока боги спят,1369,4.18
2031996,0.870530,787433,41415351,Talk Is Cheap,2668,4.78
2031997,0.870463,787433,18700925,Breaking Skin,1056,3.97
2031998,0.869393,787433,68126848,Muffins In The Freezer,4376,5.27
2031999,0.868437,787433,25996084,Полетела душа,1525,4.27
2032000,0.857681,787433,632055,Heartbeat,1425,4.22
2032001,0.855658,787433,40351129,Феникс,1264,4.12
2032002,0.855346,787433,787430,Hell Yeah,6670,5.73
2032003,0.854704,787433,49923427,Monster,1949,4.49


In [16]:
df_album = pd.DataFrame(df_items['album'].apply(eval).explode('album').tolist()).rename(columns={'track_id': 'item_id'})

album_encoder = sklearn.preprocessing.LabelEncoder()
album_encoder.fit(df_album['id'])

df_album['id_enc'] = album_encoder.transform(df_album['id'])
df_album['item_id_enc'] = item_encoder.transform(df_album['item_id'])

als_model = get_als_model(df_album['rating'], df_album['id_enc'], df_album['item_id_enc'])
similar_items_album = get_similar_items(als_model, df_album['item_id_enc'].unique())

100%|██████████| 10/10 [00:02<00:00,  4.79it/s]


In [17]:
print_sim_items(44104376, similar_items_album)

,item_id,name,votes,rating
24623,44104376,Niente da dire,1103,4.01


,score,item_id_1,item_id_2,name,votes,rating
1869507,1.000000,44104376,44104369,Fear for Nobody,9130,6.09
1869508,1.000000,44104376,44104365,L'altra dimensione,4953,5.40
1869509,1.000000,44104376,44104374,Are You Ready?,2861,4.84
1869511,1.000000,44104376,44104375,Close to the Top,1067,3.98
1869512,1.000000,44104376,44104370,Le parole lontane,3180,4.94
1869513,1.000000,44104376,44104364,New Song,5274,5.47
1869514,1.000000,44104376,44104372,Lasciami stare,1364,4.18
1869515,1.000000,44104376,39918391,Morirò da Re,7339,5.84
1869516,1.000000,44104376,43589177,Torna a casa,4068,5.19
1869517,0.673349,44104376,20690990,Дура,2244,4.61


In [18]:
similar_items = pd.concat([
    similar_items_users,
    similar_items_genre,
    similar_items_artist,
    similar_items_album
])

In [19]:
similar_items['rank'] = similar_items.groupby(['item_id_1', 'item_id_2']).cumcount(ascending=False) + 1

In [20]:
similar_items.shape

(7972785, 4)

In [21]:
similar_items_1 = similar_items.groupby(['item_id_1', 'item_id_2']).agg(score=('score', 'sum'), rank=('rank', 'max')).reset_index()

similar_items_1['score'] = similar_items_1['score'] / 4
similar_items_1.sort_values('score', ascending=False, inplace=True)
similar_items_1['rank_score'] = similar_items_1.groupby(['item_id_1']).cumcount(ascending=True) + 1

In [26]:
similar_items_1.query('item_id_1 == 96366203')

,item_id_1,item_id_2,score,rank,rank_score
7644327,96366203,96366196,0.999775,4,1
7644325,96366203,96366194,0.749875,3,2
7644324,96366203,96366193,0.749759,3,3
7644331,96366203,96366204,0.749685,3,4
7644330,96366203,96366201,0.749670,3,5
...,...,...,...,...,...
7644236,96366203,39166172,0.122798,1,154
7644193,96366203,21825965,0.122197,1,155
7644194,96366203,21825967,0.122197,1,156
7644195,96366203,21825969,0.122197,1,157


In [25]:
similar_items_1

,item_id_1,item_id_2,score,rank,rank_score
7644327,96366203,96366196,0.999775,4,1
7643688,96366196,96366203,0.999775,4,1
5392493,52974555,52974552,0.999663,4,1
2433562,24183553,24183545,0.999654,4,1
5392838,52974557,52974561,0.999651,4,1
...,...,...,...,...,...
1964900,18006276,31998691,0.072178,1,196
1964917,18006276,39450215,0.072177,1,197
1964823,18006276,14380434,0.072172,1,198
1964857,18006276,19673912,0.072172,1,199
